<a href="https://colab.research.google.com/github/mollybocock/AI-Assignment-1/blob/main/Semantic_Mirrors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semantic Mirrors
## Exploring Oppositeness in Human and LLM Embedding Space

## Project Overview

The goal of this project is to examine oppositeness in sentences and map them onto the embedding space to provide insight into how humans and LLMs understand opposites, and how this compares to vectorized opposites in the embedding space.

LLMs are generally aligned with humans when asking for the opposite of short, simple sentences (e.g. "I love bananas" → "I hate bananas"). For longer sentences, the LLM asked if the opposite sentence should be opposite in feeling or by providing antonyms to individual words. The LLM was less successful identifying opposites by cultural reference.

What the "opposite" of something is vs. what is the greatest distance away are two separate ideas that made it challenging to come up with an unbiased question not designed to trick human participants or an LLM. LLMs understand capitalized and lowercase letters differently, whereas this is unlikely a consideration for humans.

26 human survey responses were generated and respondents were asked to provide an opposite word for each word in the sentence **"The quick brown fox jumps over the lazy dog."** This sentence is simple enough for everyone to understand, uses all letters in the English alphabet, and is not so short as to not generate interesting results. The human generated responses, the LLM (GPT-3.5-turbo) generated response, and the original sentence were vectorized using OpenAI's text-embedding-3-small API and then mapped using t-SNE and UMAP.

**Findings:** The opposite sentence farthest from the original was generated by a human, not the LLM. The scatter of human opposites shows that "opposite" is subjective and context-dependent, even with words that had near-consensus responses (dog→cat, quick→slow). The LLM appears to give a general approximation of oppositeness aligned with its understanding of human perspective — finding significant vector distance from the original, but not the maximum possible distance. It appears the LLM wants sentences to be distinct but not too far from what a human might say. A limitation is that the prompt asked for individual words to keep uniformity, which constrains an LLM that might have very different "opposites" in its embedding space. Overall, humans display a more nuanced concept of opposites while LLMs adopt a common-ground approach aligned with perceived human understanding.

## 1. Setup: Libraries and API Key

In [ ]:
# Install required packages
!pip install --upgrade openai pandas matplotlib seaborn scikit-learn umap-learn scipy

In [ ]:
from openai import OpenAI
from google.colab import userdata

# Instantiate the client with your key (stored in Colab Secrets as "Secret-Key")
client = OpenAI(api_key=userdata.get("Secret-Key"))

# Test chat completion
try:
    resp = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": "Hello!"}],
        max_tokens=5
    )
    print("✅ Chat OK:", resp.choices[0].message.content.strip())
except Exception as e:
    print("❌ Chat failed:", e)

# Test embedding
try:
    emb = client.embeddings.create(
        model="text-embedding-3-small",
        input="Test embedding"
    ).data[0].embedding
    print("✅ Embedding length:", len(emb))
except Exception as e:
    print("❌ Embedding failed:", e)

## 2. Upload Survey Data

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload human_survey_responses.csv when prompted

In [ ]:
import pandas as pd

survey_df = pd.read_csv('human_survey_responses.csv')
# Strip trailing spaces from column names (Google Forms artifact)
survey_df.columns = survey_df.columns.str.strip()
print("Columns:", survey_df.columns.tolist())
survey_df.head()

## 3. Preprocessing and Addressing Capitalization

The first time I ran code to define the LLM opposites from ChatGPT I noticed that "the" only appeared once. Python dictionaries can't have duplicate keys, so if a dictionary is defined with the same key twice only the last value is kept. Python dictionaries are also case sensitive.

OpenAI's embeddings are very case sensitive ([source](https://community.openai.com/t/is-openai-chatgpt-tokenization-case-sensitive/231108/2)) and punctuation sensitive ([source](https://community.openai.com/t/embedding-very-sensitive-to-punctuation/546205)). I removed the period "." from the original sentence to reduce bias — asking for the "opposite" of an input is not the same as asking what is farthest away from it. Asking a human "what is the greatest distance from each of these inputs" is too confusing, but asking an LLM "what is the opposite of this input" vs "what is the farthest distance away in the embedding space" generates two different answers.

In [ ]:
def preprocess_for_embedding(text):
    """Keep first word capitalization, lowercase the rest."""
    words = text.split()
    if not words:
        return ""
    return words[0] + " " + " ".join(word.lower() for word in words[1:])

# LLM opposites (ChatGPT GPT-3.5-turbo, word-by-word antonyms)
# Note: two entries for "the" — capitalized at sentence start, lowercase mid-sentence
llm_opposites = {
    "The": "A",           # Capitalized — sentence start
    "quick": "slow",
    "brown": "white",
    "fox": "dog",
    "jumps": "falls",
    "over": "under",
    "the": "a",           # Lowercase — mid-sentence
    "lazy": "energetic",
    "dog": "cat"
}

print("Semantic mirror opposites:")
for word, opposite in llm_opposites.items():
    print(f"  {word} → {opposite}")

## 4. Opposites Generator

This section defines all the logic for generating opposite sentences from the survey data and LLM mapping. This was originally maintained as a separate `opposites_generator.py` script; it is inlined here so the notebook is fully self-contained.

In [ ]:
import pandas as pd
import numpy as np

# ── Original sentence ──────────────────────────────────────────────────────────
ORIGINAL_SENTENCE = "The quick brown fox jumps over the lazy dog"

# ── LLM mapping (GPT-3.5-turbo word-by-word antonyms) ─────────────────────────
LLM_MAPPING = {
    "The": "A", "quick": "slow", "brown": "white", "fox": "dog",
    "jumps": "falls", "over": "under", "the": "a",
    "lazy": "energetic", "dog": "cat"
}

# ── Human survey responses (26 respondents, collected April 2025) ──────────────
HUMAN_RESPONSES = [
    ["4/12/2025 10:21:31", "A", "Slow", "Red", "Dog", "Crawl", "Under", "A", "Hard working", "Cat"],
    ["4/12/2025 12:00:34", "An", "Slow", "Green", "Dog", "Stands", "Under", "An", "Not lazy", "Cat"],
    ["4/12/2025 12:00:41", "Start", "Slow", "Red", "Hound", "Sits", "Under", "Start", "Active", "Cat"],
    ["4/12/2025 12:09:01", "A", "Slow", "White", "Sloth", "Sleeps", "Under", "A", "Energetic", "Cat"],
    ["4/12/2025 12:15:22", "A", "Slow", "Black", "Rabbit", "Walks", "Under", "An", "Quick", "Cat"],
    ["4/12/2025 12:51:54", "Of", "Slow", "Black", "Bird", "Rolls", "Under", "Of", "Active", "Cat"],
    ["4/12/2025 14:14:56", "A", "Slow", "Pink", "Bird", "Lies down", "Under", "A", "Determined", "Cat"],
    ["4/12/2025 14:37:42", "And", "Slow", "Blue", "", "Sinks", "Under", "And", "Spry", "Cat"],
    ["4/12/2025 14:46:43", "Eht", "Slow", "White", "Hound", "Crawls", "Start", "Eht", "Determined", "Cat"],
    ["4/12/2025 14:47:12", "Not", "Fast", "Black", "Hou d", "Leaps", "Under", "Price", "Animated", "Car"],
    ["4/12/2025 15:00:22", "A", "Slow", "Black", "Red Panda", "Slides", "Under", "A", "Quick", "Cat"],
    ["4/12/2025 15:26:42", "Any", "Slow", "Yellow", "Hound", "Falls", "Under", "Any", "Industrious", "Cat"],
    ["4/12/2025 15:32:43", "Every", "Fast", "Purple", "Whale", "Stands", "Under", "Every", "Proactive", "Cat"],
    ["4/12/2025 15:51:44", "A", "Slow", "Black", "Dog", "Stands", "Under", "A", "Motivated", "Cat"],
    ["4/12/2025 16:26:25", "Any", "Slow", "Clear", "Eagle", "Dives", "Under", "Any", "Motivated", "Cat"],
    ["4/12/2025 17:08:08", "It", "Slow", "Yellow", "Tortoise", "Digs", "Under", "It", "Active", "Cat"],
    ["4/12/2025 18:11:50", "Any", "Slow", "Blue", "Hound", "Tunnels", "Under", "Any", "hardworking", "Cat"],
    ["4/12/2025 18:29:35", "A", "Slow", "White", "Not fox", "Crouch", "Under", "A", "Active", "Cat"],
    ["4/12/2025 18:32:50", "And", "Slow", "Red", "Hound", "Crawls", "Under", "And", "Productive", "Cat"],
    ["4/12/2025 18:45:28", "How", "Slow", "Green", "Dog", "Stands", "Under", "How", "Active", "Cat"],
    ["4/12/2025 18:50:24", "Those", "Slow", "Pink", "Elephant", "Stops", "Under", "Those", "Motivated", "Cat"],
    ["4/12/2025 18:51:58", "A", "Slow", "Gray", "Wildcat", "Ducks", "Under", "A", "Hardworking", "Cat"],
    ["4/12/2025 19:15:17", "A", "Slow", "White", "Hare", "Sits", "Under", "A", "Active", "Cat"],
    ["4/12/2025 21:01:51", "Eht", "Slow", "White", "Hound", "Crawls", "Start", "Eht", "Determined", "Cat"],
    ["4/13/2025 11:20:34", "A/An", "Slow", "Blue", "Sheep", "Dives", "Under", "A/An", "Energetic", "Cat"],
    ["4/13/2025 18:40:34", "a", "Slow", "White", "Turtle", "Falls", "Under", "A", "Quick", "Cat"],
]

def generate_llm_sentence():
    words = ORIGINAL_SENTENCE.split()
    return " ".join(LLM_MAPPING.get(w, w) for w in words)

def generate_capitalization_aware_sentence():
    """LLM opposite with position-aware capitalization."""
    return preprocess_for_embedding(generate_llm_sentence())

def get_consensus_opposite_sentence():
    """Most common human opposite for each word position."""
    consensus = []
    for col_idx in range(1, 10):
        vals = [r[col_idx] for r in HUMAN_RESPONSES if col_idx < len(r) and r[col_idx]]
        counter = {}
        for v in vals:
            counter[v] = counter.get(v, 0) + 1
        most_common = max(counter.items(), key=lambda x: x[1])[0] if counter else ""
        consensus.append(most_common)
    return preprocess_for_embedding(" ".join(consensus))

def generate_all_opposite_sentences():
    human_sentences = []
    for response in HUMAN_RESPONSES:
        words = [response[i] for i in range(1, 10)]
        sentence = " ".join([w for w in words if w])
        human_sentences.append(sentence)
    return {
        "original": ORIGINAL_SENTENCE,
        "llm": [generate_llm_sentence()],
        "human": human_sentences
    }

def get_all_data_as_dataframe():
    data = [{"source": "Original", "timestamp": "", "sentence": ORIGINAL_SENTENCE}]
    data.append({"source": "LLM", "timestamp": "", "sentence": generate_llm_sentence()})
    for response in HUMAN_RESPONSES:
        words = [response[i] for i in range(1, 10)]
        sentence = " ".join([w for w in words if w])
        data.append({"source": "Human", "timestamp": response[0], "sentence": sentence})
    return pd.DataFrame(data)

def analyze_opposite_word_distributions():
    original_words = ORIGINAL_SENTENCE.lower().split()
    results = {}
    for i, word in enumerate(original_words):
        col_idx = i + 1
        vals = [r[col_idx] for r in HUMAN_RESPONSES if col_idx < len(r) and r[col_idx]]
        counter = {}
        for v in vals:
            counter[v] = counter.get(v, 0) + 1
        sorted_items = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        most_common = sorted_items[0] if sorted_items else ("", 0)
        results[word] = {
            "unique_opposites": len(counter),
            "most_common": most_common[0],
            "most_common_pct": f"{most_common[1]/len(vals)*100:.1f}%" if vals else "0%",
            "all_opposites": dict(sorted_items)
        }
    return results

# Preview
sentences = generate_all_opposite_sentences()
print(f"Original:  {sentences['original']}")
print(f"LLM:       {sentences['llm'][0]}")
print(f"Cap-Aware: {generate_capitalization_aware_sentence()}")
print(f"Consensus: {get_consensus_opposite_sentence()}")
print(f"\nHuman examples (first 3):")
for s in sentences['human'][:3]:
    print(f"  {s}")

print("\nWord opposition analysis:")
analysis = analyze_opposite_word_distributions()
for word, stats in analysis.items():
    print(f"  '{word}' → most common: '{stats['most_common']}' ({stats['most_common_pct']}) | {stats['unique_opposites']} unique responses")

## 5. Build Master Sentence DataFrame

In [ ]:
# Assemble all sentences into one dataframe
rows = (
    [{"source": "Original",        "sentence": ORIGINAL_SENTENCE}] +
    [{"source": "LLM",             "sentence": generate_llm_sentence()}] +
    [{"source": "LLM (Cap-Aware)", "sentence": generate_capitalization_aware_sentence()}] +
    [{"source": "Consensus",       "sentence": get_consensus_opposite_sentence()}] +
    [{"source": "Human",           "sentence": preprocess_for_embedding(s)}
     for s in generate_all_opposite_sentences()["human"]]
)
sentences_df = pd.DataFrame(rows).reset_index(drop=True)

print(f"Total sentences: {len(sentences_df)}")
print(sentences_df["source"].value_counts().to_string())
print("\nSample sentences:")
print(sentences_df.head(6).to_string(index=False))

## 6. Generate OpenAI Embeddings

Each sentence is embedded using OpenAI's `text-embedding-3-small` model (1536 dimensions). Results are saved to `data/sentence_embeddings.pkl` so the API calls only need to be made once.

In [ ]:
import numpy as np
import time
import os

EMBEDDING_MODEL = "text-embedding-3-small"

def get_embedding(text, retries=3):
    for attempt in range(retries):
        try:
            response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
            return np.array(response.data[0].embedding)
        except Exception as e:
            print(f"  ⚠️  Attempt {attempt+1} failed: {e}")
            time.sleep(2)
    return None

print(f"Generating embeddings for {len(sentences_df)} sentences...")
print(f"Model: {EMBEDDING_MODEL} (1536 dimensions)\n")

embeddings = []
for i, row in sentences_df.iterrows():
    print(f"[{i+1}/{len(sentences_df)}] {row['source']}: {row['sentence'][:60]}...")
    emb = get_embedding(row["sentence"])
    embeddings.append(emb)
    time.sleep(0.1)

sentences_df["embedding"] = embeddings

failed = sentences_df[sentences_df["embedding"].isna()]
if len(failed) > 0:
    print(f"\n⚠️  {len(failed)} embeddings failed")
else:
    print(f"\n✅ All {len(sentences_df)} embeddings generated successfully")
    print(f"Embedding dimensionality: {len(sentences_df['embedding'].iloc[0])}")

os.makedirs("data", exist_ok=True)
sentences_df.to_pickle("data/sentence_embeddings.pkl")
print("Saved to data/sentence_embeddings.pkl")

## 7. Cosine Similarity Analysis

Cosine similarity measures how close two sentences are in embedding space. A lower value means more semantically different — i.e. more "opposite." 

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs("visualizations", exist_ok=True)

embedding_matrix = np.stack(sentences_df["embedding"].values)
sim_matrix = cosine_similarity(embedding_matrix)
original_idx = sentences_df[sentences_df["source"] == "Original"].index[0]
similarities_to_original = sim_matrix[original_idx]
sentences_df["similarity_to_original"] = similarities_to_original

llm_idx = sentences_df[sentences_df["source"] == "LLM"].index[0]
llm_cap_idx = sentences_df[sentences_df["source"] == "LLM (Cap-Aware)"].index[0]
consensus_idx = sentences_df[sentences_df["source"] == "Consensus"].index[0]
human_mask = sentences_df["source"] == "Human"

llm_sim = similarities_to_original[llm_idx]
llm_cap_sim = similarities_to_original[llm_cap_idx]
consensus_sim = similarities_to_original[consensus_idx]
human_sims = similarities_to_original[human_mask]

print("=" * 55)
print("Similarity to Original Sentence")
print("=" * 55)
print(f"LLM opposite:              {llm_sim:.4f}")
print(f"LLM (Cap-Aware):           {llm_cap_sim:.4f}")
print(f"Consensus human opposite:  {consensus_sim:.4f}")
print(f"\nHuman opposites:")
print(f"  Mean:    {human_sims.mean():.4f}")
print(f"  Median:  {np.median(human_sims):.4f}")
print(f"  Min:     {human_sims.min():.4f}")
print(f"  Max:     {human_sims.max():.4f}")

pct_below_llm = (human_sims < llm_sim).mean() * 100
print(f"\nLLM is more similar to original than {pct_below_llm:.1f}% of human responses")
print(f"(i.e. LLM is less 'opposite' than {100-pct_below_llm:.1f}% of humans)")

farthest_idx = similarities_to_original.argmin()
print(f"\nFarthest sentence from original:")
print(f"  Source:     {sentences_df.loc[farthest_idx, 'source']}")
print(f"  Sentence:   {sentences_df.loc[farthest_idx, 'sentence']}")
print(f"  Similarity: {similarities_to_original[farthest_idx]:.4f}")

# ── Ranked similarity bar chart ────────────────────────────────────────────────
ranked = sentences_df[["source", "sentence", "similarity_to_original"]].copy()
ranked = ranked.sort_values("similarity_to_original")
color_map = {"Original": "#1f77b4", "LLM": "#ff7f0e",
             "LLM (Cap-Aware)": "#d62728", "Consensus": "#9467bd", "Human": "#2ca02c"}
colors = [color_map.get(s, "#2ca02c") for s in ranked["source"]]

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(range(len(ranked)), ranked["similarity_to_original"], color=colors)
ax.set_yticks(range(len(ranked)))
ax.set_yticklabels([f"{s} — {sent[:35]}..." if len(sent) > 35 else f"{s} — {sent}"
                    for s, sent in zip(ranked["source"], ranked["sentence"])], fontsize=7)
ax.set_xlabel("Cosine Similarity to Original", fontsize=12)
ax.set_title("Sentence Similarity to Original (lower = more opposite)", fontsize=14)
ax.axvline(llm_sim, color="#ff7f0e", linestyle="--", linewidth=1.5, label="LLM similarity")
ax.legend()
plt.tight_layout()
plt.savefig("visualizations/similarity_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Pairwise heatmap ───────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(14, 12))
sns.heatmap(sim_matrix, ax=ax2, cmap="coolwarm", vmin=0.3, vmax=1.0,
            xticklabels=False, yticklabels=False)
ax2.set_title("Pairwise Cosine Similarity Heatmap", fontsize=14)
plt.tight_layout()
plt.savefig("visualizations/pairwise_similarities.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved visualizations to visualizations/")

## 8. t-SNE Visualization

t-SNE reduces the 1536-dimensional embeddings to 2D and 3D for visualization. The convex hull drawn around human responses shows the spread of human intuitions about oppositeness.

In [ ]:
from sklearn.manifold import TSNE
from scipy.spatial import ConvexHull
from mpl_toolkits.mplot3d import Axes3D

color_map = {
    "Original": "#1f77b4", "LLM": "#ff7f0e",
    "LLM (Cap-Aware)": "#d62728", "Consensus": "#9467bd", "Human": "#2ca02c"
}
size_map = {"Original": 300, "LLM": 250, "LLM (Cap-Aware)": 250, "Consensus": 200, "Human": 80}
PERPLEXITY = min(10, len(sentences_df) - 1)

# ── 2D t-SNE ───────────────────────────────────────────────────────────────────
print("Running 2D t-SNE...")
tsne_2d = TSNE(n_components=2, perplexity=PERPLEXITY, max_iter=2000, random_state=42)
result_2d = tsne_2d.fit_transform(embedding_matrix)

fig, ax = plt.subplots(figsize=(13, 10), facecolor="black")
ax.set_facecolor("black")

for source, color in color_map.items():
    mask = sentences_df["source"] == source
    ax.scatter(result_2d[mask, 0], result_2d[mask, 1],
               c=color, s=size_map[source], alpha=0.9,
               edgecolors="white", linewidths=0.5, label=source, zorder=3)

# Convex hull around human responses
human_mask = sentences_df["source"] == "Human"
human_points = result_2d[human_mask]
if len(human_points) >= 3:
    hull = ConvexHull(human_points)
    for simplex in hull.simplices:
        ax.plot(human_points[simplex, 0], human_points[simplex, 1],
                color="#2ca02c", alpha=0.3, linewidth=1)

# Annotate non-human points
for idx, row in sentences_df[sentences_df["source"] != "Human"].iterrows():
    ax.annotate(row["source"], (result_2d[idx, 0], result_2d[idx, 1]),
                fontsize=8, color="white", xytext=(6, 4), textcoords="offset points")

# Annotate farthest human point
farthest_sentence = sentences_df.loc[similarities_to_original.argmin(), "sentence"]
farthest_mask = sentences_df["sentence"] == farthest_sentence
farthest_idx_2d = sentences_df[farthest_mask].index[0]
ax.annotate("← Farthest from original",
            (result_2d[farthest_idx_2d, 0], result_2d[farthest_idx_2d, 1]),
            fontsize=8, color="#17becf",
            xytext=(8, 4), textcoords="offset points",
            arrowprops=dict(arrowstyle="->", color="#17becf", lw=1.2))

ax.set_title("2D t-SNE: Sentence Embeddings in OpenAI Embedding Space",
             fontsize=15, color="white", pad=15)
ax.set_xlabel("t-SNE Dimension 1", fontsize=11, color="white")
ax.set_ylabel("t-SNE Dimension 2", fontsize=11, color="white")
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.legend(loc="upper right", fontsize=9, framealpha=0.3, facecolor="grey", labelcolor="white")
plt.tight_layout()
plt.savefig("visualizations/embedding_space_tsne_2d.png", dpi=150,
            bbox_inches="tight", facecolor="black")
plt.show()
print("Saved: visualizations/embedding_space_tsne_2d.png")

# ── 3D t-SNE ───────────────────────────────────────────────────────────────────
print("\nRunning 3D t-SNE...")
tsne_3d = TSNE(n_components=3, perplexity=PERPLEXITY, max_iter=2000, random_state=42)
result_3d = tsne_3d.fit_transform(embedding_matrix)

sentences_df["tsne_x"] = result_3d[:, 0]
sentences_df["tsne_y"] = result_3d[:, 1]
sentences_df["tsne_z"] = result_3d[:, 2]

fig3d = plt.figure(figsize=(14, 11), facecolor="black")
ax3d = fig3d.add_subplot(111, projection="3d")
ax3d.set_facecolor("black")

for source, color in color_map.items():
    mask = sentences_df["source"] == source
    ax3d.scatter(result_3d[mask, 0], result_3d[mask, 1], result_3d[mask, 2],
                 c=color, s=size_map[source], alpha=0.9,
                 edgecolors="white", linewidths=0.3, label=source, depthshade=True)

ax3d.set_title("3D t-SNE: Sentence Embeddings", fontsize=15, color="white", pad=15)
ax3d.set_xlabel("Dim 1", fontsize=10, color="white")
ax3d.set_ylabel("Dim 2", fontsize=10, color="white")
ax3d.set_zlabel("Dim 3", fontsize=10, color="white")
ax3d.tick_params(colors="white")
ax3d.xaxis.pane.fill = False
ax3d.yaxis.pane.fill = False
ax3d.zaxis.pane.fill = False
ax3d.xaxis.pane.set_edgecolor("none")
ax3d.yaxis.pane.set_edgecolor("none")
ax3d.zaxis.pane.set_edgecolor("none")
ax3d.view_init(elev=25, azim=45)
ax3d.legend(loc="upper left", fontsize=9, framealpha=0.3, facecolor="grey", labelcolor="white")
plt.tight_layout()
plt.savefig("visualizations/embedding_space_tsne_3d.png", dpi=150,
            bbox_inches="tight", facecolor="black")
plt.show()
print("Saved: visualizations/embedding_space_tsne_3d.png")

sentences_df.to_pickle("data/sentence_embeddings.pkl")
print("\nUpdated data/sentence_embeddings.pkl with t-SNE coordinates")

## 9. UMAP Visualization

In [ ]:
from umap import UMAP

print("Running UMAP...")
reducer = UMAP(n_components=2, n_neighbors=10, min_dist=0.3, random_state=42)
umap_result = reducer.fit_transform(embedding_matrix)

sentences_df["umap_x"] = umap_result[:, 0]
sentences_df["umap_y"] = umap_result[:, 1]

fig, ax = plt.subplots(figsize=(13, 10), facecolor="black")
ax.set_facecolor("black")

for source, color in color_map.items():
    mask = sentences_df["source"] == source
    ax.scatter(umap_result[mask, 0], umap_result[mask, 1],
               c=color, s=size_map[source], alpha=0.9,
               edgecolors="white", linewidths=0.5, label=source, zorder=3)

# Convex hull around human responses
human_points_umap = umap_result[sentences_df["source"] == "Human"]
if len(human_points_umap) >= 3:
    hull = ConvexHull(human_points_umap)
    for simplex in hull.simplices:
        ax.plot(human_points_umap[simplex, 0], human_points_umap[simplex, 1],
                color="#2ca02c", alpha=0.3, linewidth=1)

for idx, row in sentences_df[sentences_df["source"] != "Human"].iterrows():
    ax.annotate(row["source"], (umap_result[idx, 0], umap_result[idx, 1]),
                fontsize=8, color="white", xytext=(6, 4), textcoords="offset points")

ax.set_title("UMAP: Sentence Embeddings in OpenAI Embedding Space",
             fontsize=15, color="white", pad=15)
ax.set_xlabel("UMAP Dimension 1", fontsize=11, color="white")
ax.set_ylabel("UMAP Dimension 2", fontsize=11, color="white")
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.legend(loc="upper right", fontsize=9, framealpha=0.3, facecolor="grey", labelcolor="white")
plt.tight_layout()
plt.savefig("visualizations/embedding_space_umap.png", dpi=150,
            bbox_inches="tight", facecolor="black")
plt.show()
print("Saved: visualizations/embedding_space_umap.png")

sentences_df.to_pickle("data/sentence_embeddings.pkl")
print("Updated data/sentence_embeddings.pkl with UMAP coordinates")

## 10. Interactive Streamlit App

An interactive version of this analysis is available as `app.py` in this repository. It provides:
- **Embedding Space** tab: 2D t-SNE, 3D t-SNE, and UMAP visualizations
- **Similarity Analysis** tab: Ranked bar chart and key statistics
- **Sentence Explorer** tab: Browse all sentences with similarity scores
- **Try Your Own** tab: Input your own opposite sentence and see where it lands

To run locally:
```bash
pip install -r requirements.txt
export OPENAI_API_KEY=your_key_here
streamlit run app.py
```